In [ ]:
import pandas as pd
import numpy as np
import glob
import os

In [ ]:
def analyze_data(df, keys, target_col="TARGET"):
    analysis = []
    for col in df.columns:
        if col in keys or col in ['_ID_']:
            continue
        data = df[col]
        col_info = {
            "column": col,
            "dtype": data.dtype,
            "null_count": data.isnull().sum(),
            "null_ratio": data.isnull().mean(),
            "distinct_count": data.nunique()
        }
        if pd.api.types.is_numeric_dtype(data):
            col_info["zero_count"]    = (data == 0).sum()
            col_info["zero_ratio"]    = (data == 0).mean()
            col_info["min_incl_0"]    = data.min()
            col_info["max_incl_0"]    = data.max()
            col_info["median_incl_0"] = data.median()
            col_info["median_excl_0"] = data[data != 0].median()
            col_info["p1_incl_0"]     = data.quantile(0.01)
            col_info["p5_incl_0"]     = data.quantile(0.05)
            col_info["p95_incl_0"]    = data.quantile(0.95)
            col_info["p99_incl_0"]    = data.quantile(0.99)
        else:
            col_info["zero_count"]    = np.nan
            col_info["zero_ratio"]    = np.nan
            col_info["min_incl_0"]    = np.nan
            col_info["max_incl_0"]    = np.nan
            col_info["median_incl_0"] = np.nan
            col_info["median_excl_0"] = np.nan
            col_info["p1_incl_0"]     = np.nan
            col_info["p5_incl_0"]     = np.nan
            col_info["p95_incl_0"]    = np.nan
            col_info["p99_incl_0"]    = np.nan

        if col != target_col and target_col in df.columns:
            grouped = df.groupby(target_col)[col]
            for group_name, group_data in grouped:
                col_info[f"null_ratio_target_{group_name}"]     = group_data.isnull().mean()
                col_info[f"distinct_count_target_{group_name}"] = group_data.nunique()
                if pd.api.types.is_numeric_dtype(group_data):
                    col_info[f"zero_ratio_target_{group_name}"]    = (group_data == 0).mean()
                    col_info[f"min_incl_0_target_{group_name}"]    = group_data.min()
                    col_info[f"max_incl_0_target_{group_name}"]    = group_data.max()
                    col_info[f"median_incl_0_target_{group_name}"] = group_data.median()
                    col_info[f"median_excl_0_target_{group_name}"] = group_data[group_data != 0].median()
                    col_info[f"p1_incl_0_target_{group_name}"]     = group_data.quantile(0.01)
                    col_info[f"p5_incl_0_target_{group_name}"]     = group_data.quantile(0.05)
                    col_info[f"p95_incl_0_target_{group_name}"]    = group_data.quantile(0.95)
                    col_info[f"p99_incl_0_target_{group_name}"]    = group_data.quantile(0.99)
                else:
                    col_info[f"zero_ratio_target_{group_name}"]    = np.nan
                    col_info[f"min_incl_0_target_{group_name}"]    = np.nan
                    col_info[f"max_incl_0_target_{group_name}"]    = np.nan
                    col_info[f"median_incl_0_target_{group_name}"] = np.nan
                    col_info[f"median_excl_0_target_{group_name}"] = np.nan
                    col_info[f"p1_incl_0_target_{group_name}"]     = np.nan
                    col_info[f"p5_incl_0_target_{group_name}"]     = np.nan
                    col_info[f"p95_incl_0_target_{group_name}"]    = np.nan
                    col_info[f"p99_incl_0_target_{group_name}"]    = np.nan

        analysis.append(col_info)
    return pd.DataFrame(analysis)

In [ ]:
def safe_cond(analysis, col, op, val):
    """Kolon yoksa False döner — OR zincirinde hiçbir satırı yanlış elemez."""
    if col not in analysis.columns:
        return pd.Series(False, index=analysis.index)
    if op == ">=":
        return analysis[col] >= val
    elif op == "==":
        return analysis[col] == val
    return pd.Series(False, index=analysis.index)


def get_alert_reasons(analysis, group_names):
    """Her satır için hangi koşuldan alerte girdiğini döner."""
    checks = [
        ('zero_ratio',     '>=', 0.99, 'zero_ratio >= 0.99'),
        ('null_ratio',     '>=', 0.99, 'null_ratio >= 0.99'),
        ('distinct_count', '==', 1,    'distinct_count == 1'),
    ]
    for gn in group_names:
        checks += [
            (f'null_ratio_target_{gn}',     '>=', 0.99, f'null_ratio_target_{gn} >= 0.99'),
            (f'zero_ratio_target_{gn}',     '>=', 0.99, f'zero_ratio_target_{gn} >= 0.99'),
            (f'distinct_count_target_{gn}', '==', 1,    f'distinct_count_target_{gn} == 1'),
        ]

    reasons = pd.Series([[] for _ in range(len(analysis))], index=analysis.index)

    for col, op, val, label in checks:
        mask = safe_cond(analysis, col, op, val)
        for idx in analysis.index[mask]:
            reasons[idx].append(label)

    return reasons.apply(lambda x: " | ".join(x) if x else "")


def sheet_name(filepath):
    return os.path.splitext(os.path.basename(filepath))[0][:31]

In [ ]:
def data_stat_func(input_folder_path, output_folder_path, keys,
                   target_col='TARGET', cat_distinct_max=500):

    source_files = glob.glob(os.path.join(input_folder_path, "*.parquet"))
    alert_list = []
    all_analysis_frames = []

    with pd.ExcelWriter(
        os.path.join(output_folder_path, 'istatistic_results.xlsx'),
        engine='openpyxl'
    ) as writer:

        for i in source_files:
            print(f'{i} başlandı.')

            # Merge yok — direkt parquet okunuyor
            df = pd.read_parquet(i)

            analysis = analyze_data(df, keys=keys, target_col=target_col)
            all_analysis_frames.append(analysis)

            group_names = list(df[target_col].dropna().unique()) if target_col in df.columns else []

            # Global alert koşulları
            alert_conditions = (
                safe_cond(analysis, 'zero_ratio',     '>=', 0.99) |
                safe_cond(analysis, 'null_ratio',     '>=', 0.99) |
                safe_cond(analysis, 'distinct_count', '==', 1)
            )

            # Target group alert koşulları (dinamik group_name)
            for gn in group_names:
                alert_conditions = (
                    alert_conditions |
                    safe_cond(analysis, f'null_ratio_target_{gn}',     '>=', 0.99) |
                    safe_cond(analysis, f'zero_ratio_target_{gn}',     '>=', 0.99) |
                    safe_cond(analysis, f'distinct_count_target_{gn}', '==', 1)
                )

            alert_vars = analysis[alert_conditions].copy()

            # Alert sebebi kolonu
            alert_vars['alert_reasons'] = get_alert_reasons(alert_vars, group_names)
            alert_vars['source_file']   = os.path.basename(i)
            alert_list.append(alert_vars)

            analysis.to_excel(writer, sheet_name=sheet_name(i), index=False)
            print(f'{i} tamamlandı.')

        if alert_list:
            pd.concat(alert_list, ignore_index=True).to_excel(
                writer, sheet_name="ALERT VARS", index=False
            )

        if all_analysis_frames:
            pd.concat(all_analysis_frames, ignore_index=True).to_excel(
                writer, sheet_name="ALL", index=False
            )

In [ ]:
# ## PARAMETRELER — kendi path ve key'lerini gir
input_folder_path  = r"input_path_buraya"
output_folder_path = r"output_path_buraya"
keys               = []   # örn. ['MUSTERI_NO', 'SOZLESME_NO']

data_stat_func(
    input_folder_path=input_folder_path,
    output_folder_path=output_folder_path,
    keys=keys,
    target_col='TARGET',
    cat_distinct_max=500
)